# ViTESI training job (Team SynapseAI)

Clones `Project-Synapse-AI/ViT-ESI-updated` at a pinned commit and runs `kaggle/run_on_kaggle.py`: environment check, regenerate the frozen data, fingerprint check, train, evaluate with the ablation harness, package.

**Running it by hand:** Settings → Accelerator **GPU T4 x2**, Internet **on**, then *Save Version → Save & Run All (Commit)* so it runs in the background. Results land in the notebook's Output (`out/`).

Any `HARD STOP` means a check failed. Don't change settings to make it pass; report it on the PR.

In [ ]:
# CONFIG (kaggle/push_job.py rewrites this cell)
REPO_URL = 'https://github.com/Project-Synapse-AI/ViT-ESI-updated.git'
COMMIT = 'set-me'
JOB = 'smoke'
MODEL = 'VIT'
TRAIN_ARGS = ''
EVAL_ARGS = ''
WIDEPEAK = False


In [ ]:
import subprocess, sys
WORK = '/tmp/vitesi'
subprocess.run(['rm', '-rf', WORK], check=True)
subprocess.run(['git', 'clone', '-q', REPO_URL, WORK], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], cwd=WORK, check=True)
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], cwd=WORK, text=True))

In [ ]:
cmd = [sys.executable, '-u', 'kaggle/run_on_kaggle.py', '--job', JOB, '--model', MODEL,
       '--train-args', TRAIN_ARGS, '--eval-args', EVAL_ARGS, '--out', '/kaggle/working/out']
if WIDEPEAK:
    cmd.append('--widepeak')
proc = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
if proc.wait() != 0:
    raise SystemExit(f'run_on_kaggle.py failed with exit code {proc.returncode}; see out/logs and out/job_manifest.json')